# song-lingo 연습 화면 (practice)

시연 화면입니다. 위에서 아래로 실행합니다. 구현된 모듈이 늘어날 때마다 셀을 추가합니다.

- 녹음은 노트북에서 직접 하지 않고 **미리 녹음한 파일 경로**를 지정합니다.
- 이 노트북의 출력에는 목소리·경로·key가 남을 수 있으므로 **커밋 전에 출력을 지우고** `scripts/scan_outputs.py`로 확인합니다.

In [ ]:
import sys
print(sys.executable)   # .../envs/voice-ai/bin/python 이어야 합니다

from dotenv import load_dotenv
load_dotenv()            # .env 의 OPENAI_API_KEY 를 읽습니다 (값은 출력하지 않습니다)

import scoring

## S0. 준비 확인

1. 채점 동작 확인 (API 호출 없음)

In [ ]:
ref = "The cat sat on the mat."
hyp = "the cat sit on mat"

r = scoring.wer(ref, hyp)
print(f"WER {r['rate']:.3f}  (치환 {r['S']}, 삭제 {r['D']}, 삽입 {r['I']} / 정답 {r['N']}단어)")
print(scoring.format_diff(r['ops']))

2. 키가 읽히는지(값은 출력하지 않음), 

In [ ]:
import os
print("키 존재:", bool(os.getenv("OPENAI_API_KEY")))
print("커널 python:", sys.executable)

In [ ]:
# import os
# from dotenv import dotenv_values, load_dotenv
# def mask(v): return "(없음)" if not v else f"{v[:7]}…{v[-4:]} (길이 {len(v)})"

# before = os.environ.get("OPENAI_API_KEY")      # 커널이 이미 들고 있는 값(load_dotenv 전)
# load_dotenv()
# print("커널의 실제 값 :", mask(os.environ.get("OPENAI_API_KEY")))
# print(".env 파일의 값 :", mask(dotenv_values(".env").get("OPENAI_API_KEY")))
# print("load_dotenv 전 :", mask(before))

3. Whisper `tiny` 모델이 로딩되는지 봅니다. 모델은 학습 저장소의 `models/`를 복사해 재사용합니다.

In [ ]:
import time
import stt   # stt.py: 녹음 찾기와 faster-whisper 인식

t0 = time.time()
model = stt.load_model()   # tiny + int8, 처음 한 번만 읽어 stt 모듈에 보관합니다
print(f"모델 로딩 성공 ({time.time() - t0:.1f}초)")

## S1. 연습 문장과 기준 음성(TTS) 준비

### 1개 생성 테스트

`sentences.csv`에서 문장을 읽고, 1번 문장의 기준 음성을 만듭니다. 모델과 voice는 013 노트북 설정(`gpt-4o-mini-tts`, `alloy`)을 따랐고, 영어가 되는지는 이 셀에서 처음 확인합니다. **실제 과금되므로** 결과를 `data/tts_cache/`에 저장해 두고 있으면 다시 호출하지 않습니다.

In [ ]:
from pathlib import Path
from IPython.display import Audio, display

import tts   # tts.py: 문장 읽기, TTS 호출과 캐시, 호출 횟수(tts.calls)
from tts import tts_wav, CACHE_DIR

sentences = tts.load_sentences()
for s in sentences:
    print(s["id"], s["sentence"])

s1 = sentences[0]
ref_path = tts_wav(s1["sentence"], CACHE_DIR / f"s{int(s1['id']):02d}.wav")
print("기준 음성:", ref_path, f"({ref_path.stat().st_size // 1024} KB), API 호출 {tts.calls}회")
display(Audio(filename=str(ref_path)))

### 연습 문장 10개의 기준 음성(TTS)

`sentences.csv`의 10문장 중 `data/tts_cache/`에 기준 음성이 없는 문장을 찾고, 필요하면 만듭니다. 문장을 고른 기준과 문장별 의도(`probe`)는 `docs/연습_문장_데이터_설명.md`에 있습니다.

- 아래 첫 셀은 **확인만** 합니다(API 호출·과금 없음).
- 두 번째 셀은 **실제로 과금**됩니다(문장 1개당 API 1회). 그래서 `GENERATE_MISSING_TTS = False`가 기본이고, `True`로 바꿔 실행하면 **캐시에 없는 문장만** TTS API로 만듭니다. 끝나면 다시 `False`로 돌려 둡니다(`scripts/check_notebook.py`로 노트북을 점검할 때 과금되지 않게 하려는 것입니다).
- 오디오 플레이어는 출력에 소리 데이터(base64)가 박히므로 **커밋 전에 출력을 지우고** `scripts/scan_outputs.py`로 확인합니다.

In [ ]:
import tts

all_sentences = tts.load_sentences()
missing_tts = [s for s in all_sentences if not (tts.CACHE_DIR / f"s{int(s['id']):02d}.wav").exists()]
print(f"문장 {len(all_sentences)}개 중 기준 음성이 없는 문장 {len(missing_tts)}개 (만들면 API 호출 {len(missing_tts)}회):")
for s in missing_tts:
    print(f"  {s['id']:>2}  {s['sentence']}   ← {s.get('probe', '')}")

In [ ]:
GENERATE_MISSING_TTS = True   # True로 바꿔 실행하면 위에서 찾은 문장만 TTS API로 만듭니다(과금). 끝나면 False로 되돌립니다.

if GENERATE_MISSING_TTS:
    from IPython.display import Audio, display
    print(f"기준 음성이 없는 문장 {len(missing_tts)}개")
    for s in missing_tts:
        path = tts.tts_wav(s["sentence"], tts.CACHE_DIR / f"s{int(s['id']):02d}.wav")   # 이미 있으면 API를 부르지 않음
        print(f"{s['id']:>2} {s['sentence']}  ({path.stat().st_size // 1024} KB)")
        display(Audio(filename=str(path)))
    print("이 커널의 API 호출 횟수:", tts.calls)
else:
    print("GENERATE_MISSING_TTS = False라서 만들지 않았습니다 (과금 없음).")

**참고: 위 확인 셀과 생성 셀의 출력이 서로 달라 보일 때**

- `tts.calls`는 **커널을 시작한 뒤 실제로 API를 부른 횟수의 누적값**입니다. 셀마다 0으로 돌아가지 않으므로, 생성이 끝난 뒤에는 어느 셀에서 찍어도 같은 값(이 프로젝트에서는 8회: 2번과 4~10번)이 나옵니다. "이번 셀에서 부른 횟수"가 아닙니다. 만들 문장이 없어 반복문이 한 번도 돌지 않았어도 이 값은 그대로입니다.
- `missing_tts`는 **확인 셀을 실행한 시점의 목록**(스냅샷)입니다. 생성 셀은 이 목록을 그대로 쓰므로, 확인 셀을 다시 실행하지 않으면 오래된 목록을 봅니다. 파일이 이미 있으면 `tts_wav`가 API를 부르지 않아 과금은 늘지 않지만 출력이 헷갈릴 수 있습니다. 그래서 **생성 셀을 돌리기 전에 확인 셀을 먼저 다시 실행**합니다.
- 각 셀의 출력은 **그 셀을 마지막으로 실행한 시점**의 값입니다. 셀 순서를 바꾸거나 일부만 다시 실행하면 위아래 출력이 서로 다른 시점의 값이 됩니다. 예: 확인 셀은 생성이 끝난 뒤에 다시 돌려 "0개", 생성 셀은 먼저 돌렸던 실행의 누적 카운터 "8".
- 10문장의 기준 음성은 2026-11-02 22:19에 8개를 만들어 모두 `data/tts_cache/`에 있습니다(파일 수정 시각으로 확인). 끝났으면 `GENERATE_MISSING_TTS`를 `False`로 되돌립니다. 그래야 `scripts/check_notebook.py`가 과금 없이 통과합니다.
- **이번 실행에서 부른 횟수만** 보려면 생성 전에 `before = tts.calls`를 저장하고, 끝난 뒤 `tts.calls - before`를 출력합니다.

## S2. 내 녹음 인식(faster-whisper)

`data/recordings/`의 `sNN_take1.*`(wav·m4a·mp3 모두 가능)를 읽어 Whisper `tiny`로 인식합니다(`language="en"`, `beam_size=1`). 노트북에서 직접 녹음하지 않고 미리 녹음한 파일을 씁니다. 이 셀의 출력에는 내 목소리의 인식 결과가 남으므로 커밋 전에 출력을 지웁니다.

In [ ]:
from stt import REC_DIR, find_recording, recognize   # stt.py로 옮긴 함수들

hyps = {}   # {문장 id: 인식 결과}
for s in sentences:
    path = find_recording(s["id"])
    if path is None:
        print(f"[{s['id']}] 녹음 없음: {REC_DIR}/s{int(s['id']):02d}_take1.*")
        continue
    hyps[s["id"]], sec = recognize(path)
    print(f"[{s['id']}] 정답: {s['sentence']}")
    print(f"    인식: {hyps[s['id']]}   ({path.suffix}, {sec:.1f}초)")

## S3. 내 녹음 채점 + 틀린 부분 표시

정답 문장과 인식 결과를 `scoring`으로 비교합니다. 점수는 발음 점수가 아니라 **문장 인식 일치도**의 오류율입니다. 오류율은 1을 넘을 수 있어 "일치율"로 바꿔 표시하지 않고 치환(S)·삭제(D)·삽입(I) 개수를 함께 보여 줍니다. 틀린 부분은 `[정답→인식]`(치환), `[-정답-]`(삭제), `[+인식+]`(삽입)으로 표시합니다.

In [ ]:
for s in sentences:
    if s["id"] not in hyps:
        continue
    w = scoring.wer(s["sentence"], hyps[s["id"]])
    c = scoring.cer(s["sentence"], hyps[s["id"]])
    print(f"문장 #{s['id']}: \"{s['sentence']}\"")
    print(f"Whisper(tiny) 인식: \"{hyps[s['id']]}\"")
    print(f"WER {w['rate']:.3f}  (치환 {w['S']}, 삭제 {w['D']}, 삽입 {w['I']} / 정답 {w['N']}단어)")
    print(f"CER {c['rate']:.3f}  (치환 {c['S']}, 삭제 {c['D']}, 삽입 {c['I']} / 정답 {c['N']}글자)")
    print(scoring.format_diff(w["ops"]))
    print()

### 대조 실험1) TTS 기준 음성을 같은 Whisper에 넣어 보기

3번 문장의 `we'll sing` 구간이 두 번 모두 틀렸습니다. 원인이 내 녹음·발음인지 `tiny` 모델의 한계인지 가르기 위해, TTS로 만든 3번 문장 기준 음성을 같은 설정으로 인식시킵니다. 표본이 1개(문장 1개, voice 1개, 1회)라서 **일반화하지 않고 이 문장에서의 참고 결과로만** 봅니다.

In [ ]:
s3 = sentences[2]
p3 = tts_wav(s3["sentence"], CACHE_DIR / f"s{int(s3['id']):02d}.wav")   # 이미 있으면 API를 부르지 않음
tts_hyp, sec = recognize(p3)
w = scoring.wer(s3["sentence"], tts_hyp)

print(f"정답          : {s3['sentence']}")
print(f"TTS 인식      : {tts_hyp}   ({sec:.1f}초, 이 커널의 API 호출 {tts.calls}회)")
print(f"내 녹음 인식  : {hyps[s3['id']]}")
print(f"TTS  WER {w['rate']:.3f} (치환 {w['S']}, 삭제 {w['D']}, 삽입 {w['I']} / {w['N']}단어)")
print(scoring.format_diff(w["ops"]))

#### 같은 문장의 TTS와 내 녹음을 소리 그림으로 비교

3번 문장(`We'll sing together until the morning comes.`)의 TTS 기준 음성과 내 녹음을 **같은 설정**으로 그립니다. 수업 코드를 가져와 썼습니다.

| 수업 내용 | 출처 | 이 그림에서의 역할 |
|---|---|---|
| WAV 읽기, 샘플링 레이트, 모노·16 kHz 변환 | 02, lesson04 4-D | 48 kHz(내 녹음)와 24 kHz(TTS)를 같은 16 kHz 모노로 맞춤 |
| STFT (창 512, 이동 128, hann) | 03 `compute_stft`, lesson04 | 시간별 주파수 분포(스펙트로그램) |
| Mel 필터 40개 + 로그 | lesson04 `logmel` | Log-Mel (Whisper가 모델 입력으로 쓰는 특징의 교육용 버전) |

**읽는 법**: 이 그림은 **참고 정보이며 평가가 아닙니다.** 스펙트로그램이 다르다고 해서 발음이 틀렸다는 뜻이 아닙니다. 두 소리의 길이·말 속도·무음 구간·에너지가 모이는 위치가 어떻게 다른지 눈으로 보는 용도입니다. 비교를 위해 두 파형은 최대 진폭을 같게 맞추고, 같은 색 범위로 그립니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import theme
theme.apply_style()   # 프로젝트 공통 색·글꼴 (theme.py, 초록 계열)

# 수업 코드를 모듈로 옮겼습니다(동작은 같음).
from audio import load_audio                  # audio.py: 어떤 녹음 파일이든 16 kHz 모노 float32 파형으로 (lesson04 4-D 순서)
from visualisation import logmel, plot_sound_pair, plot_error_spans
# visualisation.py: STFT 파워 → Mel 필터 → Log-Mel(dB) 계산(lesson04 실습 2-F·3-B)과 그림 그리기

In [ ]:
sid = sentences[2]["id"]
sources = [("TTS 기준 음성", CACHE_DIR / f"s{int(sid):02d}.wav"), ("내 녹음", find_recording(sid))]

sounds = []
for name, p in sources:
    x, sr0, ch = load_audio(p)
    x = x / max(np.abs(x).max(), 1e-9) * 0.9          # 최대 진폭을 맞춰 모양만 비교
    f = logmel(x, 16000)
    sounds.append((name, x))
    print(f"{name}: 원본 {sr0} Hz / {ch}채널 → 16000 Hz 모노, {len(x) / 16000:.2f}초, "
          f"STFT {f['power'].shape}, Log-Mel {f['log_mel'].shape}  (주파수·Mel칸, 시간)")

# 그리기는 visualisation.plot_sound_pair가 합니다(두 소리를 같은 설정·같은 색 범위로).
plot_sound_pair(sounds, f'문장 #{sid}: "{sentences[2]["sentence"]}"  (참고 정보, 평가 아님; STFT 창 512·이동 128, Mel 40, 같은 색 범위)')
plt.show()

**소리 그림 결과 검토**

표시: **[확인]** 그림에서 바로 보이는 것, **[추정]** 해석, **[미확인]** 이 그림만으로는 알 수 없는 것. 이 그림은 참고 정보이며 평가가 아닙니다.

| 관찰 | 근거 | 해석 |
|---|---|---|
| 말하는 구간의 길이는 비슷합니다. TTS는 약 0.25~3.2초(약 3초), 내 녹음은 약 1.3~4.5초(약 3.2초)입니다. | **[확인]** 파형·스펙트로그램의 소리 구간 | 말 속도가 인식 실패의 주된 원인은 아닌 것 같습니다 **[추정]**(눈대중이며 단어별 시간은 재지 않았습니다). |
| 내 녹음은 앞의 약 1.3초가 거의 무음이고, 0.8초 근처에 짧고 넓은 대역의 소리가 있습니다. | **[확인]** | 숨소리·입소리·녹음 시작 소음일 수 있습니다 **[추정]**. Whisper는 무음 뒤 첫 단어를 놓치기 쉬운지는 **[미확인]**입니다. |
| TTS의 무음은 완전히 비어 있고(가장 어두운 색), 내 녹음의 무음에는 낮은 주파수(약 0.3 kHz 이하)에 가는 띠가 남아 있습니다. | **[확인]** 스펙트로그램 아래쪽 | 배경 소음이나 마이크의 저주파 성분일 수 있습니다 **[추정]**. 계획서의 `quality.py` "80 Hz 이하 에너지 비율" 참고 항목이 이런 것을 보려는 것입니다. |
| 모음 구간에 가로 줄무늬(고조파)가 두 소리 모두에 보입니다. TTS는 줄이 곧고 굵으며, 내 녹음은 줄이 위아래로 더 많이 휘어집니다. | **[확인]** 03 STFT 수업의 줄무늬와 같은 현상 | 줄이 휘어진 정도는 음높이(억양) 변화로 볼 수 있습니다 **[일반 지식]**. 내 억양이 TTS보다 더 움직였다는 뜻인지는 **[미확인]**입니다. |
| TTS는 단어 사이에 세로 빈틈이 뚜렷하고, 내 녹음은 1.3~2.0초 구간이 이어져 빈틈이 적습니다. | **[확인]** 세로 빈틈의 선명도 | 단어가 붙어서 읽혔을 가능성이 있습니다 **[추정]**. 이 구간이 `we'll sing`인지는 단어별 시간을 재지 않아 **[미확인]**입니다. |
| 내 녹음은 말하는 동안 4~8 kHz에도 에너지가 넓게 퍼져 있고, TTS는 특정 순간(자음으로 보이는 곳)에만 높습니다. | **[확인]**(두 소리의 최대 진폭을 같게 맞춘 뒤 비교) | 마이크·녹음 환경의 차이일 수 있습니다 **[추정]**. |

**정리**: 두 소리는 눈에 띄게 다르지만, 이 그림만으로는 "인식이 틀린 이유"를 하나로 정할 수 없습니다. 다음에 가장 싸게 확인할 수 있는 것은 두 가지입니다. ① `model.transcribe(..., word_timestamps=True)`로 단어별 시간을 얻어 그림 위에 겹쳐 그리기(어느 구간이 `we'll sing`인지 확인), ② 같은 문장을 앞 무음을 줄이고 다시 녹음해 비교하기.

**색 비교 메모**: 이 그림은 처음에 수업의 `magma`(스펙트로그램)와 `Greens_r`(Log-Mel)를 섞어 썼습니다. 지금은 프로젝트 공통 초록 램프(`theme.CMAP_SPEC`)로 통일했고, 근거는 검토 문서 14.4절에 있습니다.

#### 틀린 구간을 소리 그림 위에 표시하기

S4.5의 그림은 문장 전체를 한 장으로 그린 것이라 **어느 단어가 그림의 어디인지** 알 수 없었습니다. 이번에는 채점 결과와 시간을 연결합니다.

1. 내 녹음과 TTS를 각각 `word_timestamps=True`로 인식해 **단어별 시작·끝 시간**을 얻습니다(`stt.recognize_words`, 앞 무음이 첫 단어에 섞이는 것을 줄이려고 `vad_filter=True`).
2. `scoring`의 결과에서 **연속된 틀린 칸을 한 덩어리**로 묶습니다(`align.error_blocks`). 단어 수가 달라도(`we'll` ↔ `we are`) 한 덩어리로 비교할 수 있습니다.
3. 덩어리마다 정답 단어는 **TTS의 구간**으로, 인식된 단어는 **내 녹음의 구간**으로 바꿔(`align.locate_errors`) 두 그림에 같은 번호 `[k]`로 표시합니다. 한쪽에 단어가 없으면(삭제·삽입) 이웃 단어 사이의 점선 한 줄로 표시합니다.

**읽는 법**: 이 표시는 "내가 말한 소리 중 잘못 인식된 곳이 어디이고, 같은 위치의 기준 소리는 어떻게 생겼는지"를 나란히 보는 용도입니다. **원인을 알려 주지는 않습니다.** 단어 시간은 Whisper가 모델 내부 정보로 **추정한 값**이라 정확한 강제 정렬이 아니며(`tiny`는 오차가 클 수 있음), 구간은 대략의 위치입니다.

In [ ]:
import practice as flow   # practice.py: 연습 한 번의 순서(문장 → TTS → 점검 → 인식 → 채점 → 틀린 구간). 노트북 함수는 run_practice()라 이름이 겹치지 않지만, 모듈 이름과 구분하려고 flow로 부릅니다.
import report             # report.py: 결과 dict → 읽을 글

def show_error_spans(target, take=1):
    """틀린 덩어리의 시간 구간을 TTS(왼쪽)와 내 녹음(오른쪽)의 파형·스펙트로그램·Log-Mel 위에 같은 번호로 표시한다.
    인식·구간 계산은 flow.locate_errors, 글은 report.error_blocks, 그림은 visualisation.plot_error_spans가 맡는다.
    구간은 Whisper의 단어별 시간(추정값)이라 대략의 위치이고, 평가가 아니라 참고 정보다."""
    prep = flow.prepare(target, sentences)
    rec = find_recording(prep["id"], take) if prep["id"].isdigit() else None
    if rec is None:
        print("따라 읽은 녹음이 없습니다."); return
    out = flow.locate_errors(prep, {"recording": rec})      # 인식이 2번 돌아 몇 초 걸린다
    print(report.error_blocks(prep, out["loc"]))
    plot_error_spans(out["sounds"], out["loc"]["blocks"],
                     f'문장 #{prep["id"]}: 같은 번호 [k]는 같은 덩어리 (참고 정보, 평가 아님; 구간은 Whisper 단어 시간의 추정값)')
    plt.show()

In [ ]:
# 3번 문장: 내 녹음에서 잘못 인식된 덩어리와, 같은 위치의 TTS 소리를 나란히 봅니다 (TTS는 캐시, API 호출 0회).
# 위쪽 표(구간 숫자)와 그림의 [1] 표시가 같은 곳을 가리키는지 확인하세요. 새 인식이 2번 돌아 몇 초 걸립니다.
show_error_spans(3)

### 실험2) Whisper 모델 크기 비교(같은 녹음, 여러 모델)

면담(11-02)에서 "모델을 올려 보라"는 조언을 받아, **같은 녹음을 모델 크기만 바꿔** 인식·채점해 비교합니다(판단과 근거는 검토 문서 15.2). `we'll sing`처럼 `tiny`가 계속 틀리는 구간이 큰 모델에서 달라지는지 봅니다.

- **비교할 모델은 자동으로 정합니다.** `SIZES = stt.available_models()`가 이 PC의 `models/`에 **이미 내려받아 둔 모델을 전부** 찾아 작은 모델부터 차례로 비교합니다(지금 `tiny`만 있으면 `tiny`만 돕니다). 그래서 기본 실행은 **다운로드도 과금도 없습니다.**
- 비교 대상을 늘리려면 두 가지가 있습니다. ① `DOWNLOAD_MORE = True`: `tiny`·`base`·`small` 중 아직 없는 크기도 내려받아 함께 비교합니다(인터넷이 필요하고 시간이 걸립니다. 받은 모델은 `models/`에 남고 git에서 제외됩니다). ② `SIZES`를 직접 적기: 예 `["tiny", "small", "medium"]`(`medium` 이상은 파일이 훨씬 커서 필요할 때만).
- 인식은 `stt.recognize(..., model_size=)`, 비교와 채점은 `model_compare.compare_models`가 합니다. 이 셀은 표만 보여 줍니다. 모델은 크기마다 한 번만 읽고, 읽는 시간은 인식 시간과 따로 잽니다.
- 한 모델을 읽지 못해도(내려받기 실패 등) 그 모델만 건너뛰고 나머지는 계속합니다(`건너뜀(모델을 읽지 못함)` 줄). 비교가 끝난 모델은 메모리에서 내려(`release=True`) 큰 모델을 여러 개 돌려도 메모리가 쌓이지 않게 합니다.
- 한 사람·몇 개의 녹음 결과라서 "큰 모델이 낫다"고 일반화하지 않습니다. 큰 모델은 틀린 발음을 문맥으로 더 잘 고쳐 적을 수도 있어서, 값은 **인식 일치도의 차이**로만 읽습니다(발음 점수가 아님).
- 30건 녹음 뒤에는 `IDS`, `TAKES`만 늘려 같은 셀로 모델 × 문장 × 회차 표를 만듭니다. 결과를 보고할 때는 아래의 **설정** 줄(장치·계산 방식·beam_size)과 비교한 모델 목록을 함께 적습니다.
- 출력에 내 목소리의 인식 결과가 남으므로 커밋 전에 출력을 지웁니다.

In [ ]:
import pandas as pd
from IPython.display import display

import model_compare   # model_compare.py: 같은 녹음을 여러 모델 크기로 인식·채점해 비교(크기별 인식 stt + 채점 scoring)
import stt

DOWNLOAD_MORE = True   # True면 stt.MODEL_SIZES(tiny·base·small) 중 아직 내려받지 않은 크기도 받아서 함께 비교합니다(인터넷 필요, 시간이 걸림, API 과금 없음)
SIZES = stt.available_models(include=stt.MODEL_SIZES if DOWNLOAD_MORE else None)   # 기본: 이 PC의 models/에 이미 있는 모델을 전부(작은 모델부터)
if not SIZES:
    print("models/에 내려받아 둔 모델이 없어 기본 크기(tiny)를 내려받아 씁니다.")
    SIZES = [stt.MODEL_SIZE]
IDS = [1, 2, 3]     # 비교할 문장 번호. 30건 녹음 뒤에는 list(range(1, 11))
TAKES = [1]         # 회차. 30건 녹음 뒤에는 [1, 2, 3]
print("비교할 모델:", SIZES)

# release=True: 모델 하나의 비교가 끝나면 메모리에서 내려, 큰 모델을 여러 개 돌려도 메모리가 쌓이지 않게 합니다.
cmp = model_compare.compare_models(IDS, SIZES, takes=TAKES, release=True)
print("설정:", {k: v for k, v in cmp["settings"].items() if k != "model"}, "| 모델 읽는 시간(초):", {k: round(v, 1) for k, v in cmp["load_sec"].items()})
for size, err in cmp["errors"].items():
    print("건너뜀(모델을 읽지 못함):", size, "←", err)      # 내려받기 실패 등. 이 모델만 빠지고 나머지는 계속합니다
for m in cmp["missing"]:
    print("건너뜀:", m)

df = pd.DataFrame(cmp["rows"])
if df.empty:
    print("비교할 녹음이 없습니다. data/recordings/sNN_take1.wav를 확인하세요.")
else:
    df["id"] = df["id"].astype(int)
    order = [s for s in SIZES if s in set(df["model"])]       # 표의 열 순서를 SIZES 순서로(작은 모델부터)
    cols = ["model", "vad"] if df["vad"].nunique() > 1 else "model"   # VAD 끔/켬을 둘 다 돌렸을 때만 열을 둘로 나눕니다(안 나누면 두 값이 평균으로 합쳐집니다)
    print("\nWER (오류율: 낮을수록 정답 문장에 가까움)")
    display(df.pivot_table(index=["id", "take"], columns=cols, values="wer")[order].round(3))
    print("인식 시간 (초)")
    display(df.pivot_table(index=["id", "take"], columns=cols, values="sec")[order].round(1))
    print("인식된 문장과 틀린 부분")
    view = df.assign(model=pd.Categorical(df["model"], order, ordered=True))
    display(view[["id", "take", "model", "vad", "hyp", "diff"]].sort_values(["id", "take", "model", "vad"]).reset_index(drop=True))

#### 결과 검토 (tiny·base·small, 문장 1~3의 take1 한 번씩)

표시: **[확인]** 위 출력이나 이 저장소에서 실행해 확인한 값, **[추정]** 해석, **[미확인]** 이 결과만으로는 알 수 없는 것. **한 사람·녹음 3개·1회**의 결과라서 일반화하지 않고 "이 조건에서는 이렇게 나왔다"로만 읽습니다.

**한눈에 보기**

| 문장 | tiny | base | small | 인식된 문장(3번) |
|---|---|---|---|---|
| 1번 | 0.000 | 0.000 | 0.000 | (세 모델 모두 정답과 같음) |
| 2번 | 0.000 | 0.000 | 0.000 | (세 모델 모두 정답과 같음) |
| 3번 | 0.429 | 0.429 | 0.429 | tiny `We are **seeing** together…` / base `We are **singing** together…` / small `We are **staying** together…` |

| 관찰 | 근거 | 해석 |
|---|---|---|
| **1·2번은 세 모델이 모두 맞혔습니다.** 이 두 문장으로는 모델 크기를 비교할 수 없습니다. | **[확인]** WER 0.000 ×6 | 쉬운 문장은 `tiny`로도 충분합니다 **[추정]**. 모델 비교의 변별력은 `we'll sing`처럼 어려운 구간에서 나옵니다. |
| **3번은 모델을 `tiny`→`base`→`small`로 키워도 WER이 0.429로 같고, 틀린 모양도 같습니다.** 세 모델 모두 `we'll`을 `we are`로 들었고, 달라진 것은 `sing` 자리의 단어(seeing / singing / staying)뿐입니다. | **[확인]** 세 줄 모두 `[we'll→we] [+are+] [sing→…]` | `we'll sing`을 못 맞힌 것은 **`tiny`의 용량 문제만은 아닌 것**으로 보입니다 **[추정]**. 앞의 대조 실험(TTS 음성은 `tiny`도 0.000)과 합치면 **내 녹음 쪽 요인**(발음의 연결, 마이크, 앞 무음·소음 등) 가설이 더 힘을 얻습니다 **[추정]**. 어느 요인인지는 여전히 **[미확인]**입니다. |
| **큰 모델이 틀린 발음을 문맥으로 고쳐 적지는 않았습니다.** `We'll sing together until the morning comes`는 문맥만 보면 고치기 쉬운 문장인데도 `base`·`small`은 `we are`를 그대로 적었습니다. | **[확인]** | 걱정했던 "큰 모델이 너무 관대해서 틀린 곳을 못 잡는다"는 문제는 **이 한 사례에서는 나타나지 않았습니다.** 다만 1개 문장이라 큰 모델 전반으로 말할 수는 없습니다. |
| **WER이 같아도 틀린 정도는 다릅니다.** 글자 단위 오류율(CER)은 tiny 0.135(5/37)이고 base·small은 0.162(6/37)입니다. | **[확인]** `scoring.cer`로 계산 | `sing`→`singing`은 의미로는 가까워 보이지만 글자 수로는 `seeing`보다 약간 멉니다. WER·CER 어느 쪽도 "어느 모델이 더 낫다"를 가려 주지 못합니다. **표에 CER 열을 같이 두는 것이 좋습니다.** |
| **속도: 4~5초 녹음(말한 시간은 약 2~3초) 기준으로 `base`는 `tiny`의 약 1.5~3배, `small`은 약 4~6배 걸렸습니다(컴퓨터 부하에 따라 실행마다 달라짐).** (2·3번, 이 실행: tiny 1.3~1.5초, base 3.2~3.7초, small 7.8~8.4초) | **[확인]** 인식 시간 표 | 이 PC(CPU, int8)에서는 `small`이 4~5초짜리 녹음 한 개에 5~8초 걸립니다. 웹 화면(Streamlit)에서 바로 결과를 보여 주기에는 무거울 수 있습니다 **[추정]**. |
| 1번 문장의 인식 시간(tiny 4.3초, base 4.4초)과 모델 읽는 시간(base 8.7초, small 22.0초)은 **비교에 쓰지 않습니다.** | **[확인]** | 첫 호출의 준비 시간과, 이번에 처음 **내려받은** 시간이 섞였을 가능성이 큽니다 **[추정]**. 속도는 2·3번 값으로 비교했습니다. |

**추가 확인 (위 표 밖에서 로컬로 돌려 본 값) — 모델 크기만이 아니라 인식 옵션도 결과를 바꿉니다**

위 표는 모든 모델에 **같은 옵션(`vad_filter` 끔)** 으로 돌린 것입니다. 같은 녹음을 `vad_filter=True`(말소리가 없는 구간을 걸러낸 뒤 인식)로 다시 돌려 보았습니다 **[확인]**.

| 문장 | tiny (VAD 끔 → 켬) | base (끔 → 켬) | small (끔 → 켬) |
|---|---|---|---|
| 1번 | 0.000 → 0.000 | 0.000 → 0.000 | 0.000 → 0.000 |
| 2번 | 0.000 → 0.000 | 0.000 → 0.000 | 0.000 → 0.000 |
| 3번 | 0.429 → 0.429 | 0.429 → 0.429 | **0.429 → 0.286** (`We will sing together…`) |

- **`small`만, 3번에서만** 결과가 달라졌습니다. VAD를 켜면 `we are staying`이 `we will sing`이 되어 오류가 3개에서 2개로 줄었습니다(치환 2·삽입 1 → 치환 1·삽입 1, 프로젝트의 표시로는 `[+we+] [we'll→will]`, 2/7). 인식된 단어 수는 두 경우 모두 8개입니다. 다른 문장은 그대로여서 **악화 사례는 없었습니다.** 단어 시간 옵션(`word_timestamps`)만 켠 경우는 달라지지 않았습니다.
- 단어별 확신도(`prob`)를 보면 불확실한 곳이 보입니다 **[확인]**: base `singing` **0.19**(낮음), small(VAD 켬) `will` 0.34·`sing` 0.76. 모델이 헷갈린 자리와 틀린 자리가 겹칩니다 **[추정]**.
- 이 녹음은 앞 무음이 0.8초이고 4.8초 중 말한 시간이 2.8초입니다(`quality` 점검 결과). 앞의 무음·소음이 `small`의 디코딩에 영향을 줬을 수 있습니다 **[추정]**. 녹음 한 개·한 번의 결과라서 우연일 수도 있습니다 **[미확인]**.
- **파이프라인에 주는 의미 [추정]**: 지금 채점(`practice.read`)은 VAD 없이(`stt.recognize`), 틀린 구간 표시(`practice.locate_errors`)는 VAD를 켜고(`stt.recognize_words`) 인식합니다. 기본 모델 `tiny`에서는 둘이 같지만, **`small` 이상으로 바꾸면 채점된 문장과 구간 표시의 문장이 서로 달라질 수 있습니다.** 기본 모델을 바꾸기 전에 두 경로의 옵션을 같게 맞춰야 합니다.

**결론과 다음에 할 일**

1. **기본 모델은 `tiny`를 유지합니다.** 이 3개 녹음에서는 `base`·`small`이 WER을 개선하지 못했고 약 1.5~6배 느렸습니다. 결정은 30건 녹음 뒤에 다시 합니다.
2. **30건 녹음 뒤에 같은 표를 확장합니다**(`IDS = list(range(1, 11))`, `TAKES = [1, 2, 3]`). 이때 **VAD 끔/켬을 변수로** 비교합니다(`compare_models(..., vad_filters=(False, True))`로 이미 할 수 있고, 표는 `columns=["model", "vad"]`로 펼칩니다), 표에 **CER 열**도 같이 둡니다.
3. **`we'll`의 원인 분리는 모델이 아니라 녹음 쪽에서 합니다.** 같은 문장을 (a) 앞 무음을 줄여 다시 녹음, (b) 일부러 또렷하게 `we'll`과 `we will`을 각각 읽어 비교, (c) 다른 마이크로 녹음해 봅니다.
4. 출력의 `HF_TOKEN` 경고(`unauthenticated requests`)는 Hugging Face에서 내려받을 때의 속도 제한 안내이며 동작에는 영향이 없습니다. 모델을 모두 내려받은 뒤에는 나오지 않습니다 **[일반 지식]**.

## S4. 연습 흐름: 듣고 → 따라읽기 또는 받아쓰기 → 채점

한 문장을 위에서 아래로 한 번에 연습하는 화면입니다. 지금까지 만든 부분(TTS 캐시, 녹음 점검, Whisper 인식, `scoring`)을 `practice()` 하나로 이어 붙였습니다. 이 셀은 보여 주는 일만 하고, **순서는 `practice.py`, 글은 `report.py`** 가 맡습니다(같은 함수를 나중에 웹 화면에서도 부를 수 있도록 모듈로 분리).

| 모드 | 순서 |
|---|---|
| `"read"` 따라 읽기 | 문장과 뜻 확인 → 기준 음성 듣기 → **녹음 상태 점검** → (미리 녹음한 파일을) 인식 → 오류율과 틀린 부분 |
| `"dictation"` 받아쓰기 | 기준 음성 듣기(문장은 숨김) → 들은 대로 입력 → 오류율과 틀린 부분 → 정답과 뜻 공개 |

- 문장은 `sentences.csv`의 번호(`1`, `2`, `3`)로 부르거나, 직접 쓴 영어 문장을 문자열로 넣을 수 있습니다.
- **직접 입력한 문장은 TTS API로 전송되고 과금됩니다**(한 번 만든 음성은 `data/tts_cache/`에 저장해 다시 쓰지 않습니다). **직접 쓴 문장만 넣고, 상업 노래의 가사는 넣지 않습니다.**
- 점수는 발음 점수가 아니라 문장 인식 일치도의 오류율입니다. 받아쓰기는 STT를 거치지 않아 인식 오류가 섞이지 않고, 맞춤법·띄어쓰기 같은 내 쓰기 실수만 점수에 반영됩니다.
- 녹음 상태 점검의 경고는 안내일 뿐이며, 경고가 있어도 인식과 채점은 그대로 진행합니다.
- 받아쓰기의 `input()` 창은 VS Code에서 화면 위쪽에 나타납니다.

In [ ]:
def run_practice(target, mode="read", recording=None):
    """한 문장 연습. target: sentences.csv의 id 또는 직접 쓴 문장.
    mode="read": 문장 확인 → 듣기 → 녹음 점검 → 인식 → 채점 (recording을 안 주면 data/recordings/sNN_take1.*)
    mode="dictation": 듣기(문장은 숨김) → 들은 대로 입력 → 채점 → 정답 공개
    이 셀은 화면에 보여 주는 일만 한다. 순서는 practice.py(flow), 글은 report.py가 만든다."""
    if mode not in ("read", "dictation"):
        raise ValueError('mode는 "read" 또는 "dictation"입니다.')
    prep = flow.prepare(target, sentences)            # 문장 찾기 + 기준 음성(TTS, 캐시)
    if prep["error"]:
        print(prep["error"])
        return
    print(report.header(prep, mode))
    print(report.tts_status(prep))
    display(Audio(filename=str(prep["tts_path"])))

    if mode == "read":
        res = flow.read(prep, recording)               # 녹음 점검 → 인식 → 채점
    else:
        res = flow.dictation(prep, input("들은 대로 적어 보세요 > "))
    print(report.result(prep, res))

In [ ]:
# 1) 따라 읽기: 3번 문장을 듣고, 미리 녹음한 data/recordings/s03_take1.wav를 인식해 채점합니다 (TTS는 캐시, 호출 0회).
run_practice(3, "read")

In [ ]:
# 2) 받아쓰기: 1번 문장을 듣고 들은 대로 입력합니다 (입력창은 화면 위쪽에 뜹니다. TTS는 캐시, 호출 0회).
run_practice(1, "dictation")

In [ ]:
# 3) 직접 쓴 문장으로 받아쓰기: 문장을 바꿔 쓸 수 있습니다. 처음 쓰는 문장은 TTS를 새로 만들어 과금(1회)됩니다.
#    직접 쓴 문장만 넣고, 상업 노래의 가사는 넣지 않습니다.
run_practice("Where the river meets the sea.", "dictation")

In [ ]:
# 2026.11.1(일) 21h30 __ 테스트
import quality
print(quality.format_report(quality.check_file("data/recordings/s01_take1.wav")))

In [ ]:
import quality, glob
print(f"{'파일':14s}{'길이':>6s}{'RMS':>8s}{'최대':>6s}{'클리핑':>7s}{'무음':>6s}  경고")
for p in sorted(glob.glob("data/quality_test/*.wav")):
    r = quality.check_file(p); m = r["metrics"]
    print(f"{p.split('/')[-1]:14s}{m['seconds']:5.1f}s{m['rms_dbfs']:7.1f}{m['peak']:6.2f}{m['clip_ratio']*100:6.2f}%{m['silent_ratio']*100:5.0f}%  "
        + (" / ".join(w[:9] for w in r["warnings"]) or "없음"))